In [1]:
import pandas as pd
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from forecast_checks import LeakageChecker, ForecastValidator

In [ ]:
DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "dataset.csv", parse_dates=["date"])
df["location"] = df["location"].astype("category") #brnahc on location

In [ ]:
#use an 80/20 split
min_date = df.min("time")
max_date = df.max("time")
cutoff_date = Math.floor(max_date.timeDelta(min_date)*0.8).todate()

train = df[df["date"] < cutoff_date - pd.Timedelta(days=6)]
print(train.shape[0]/df.shape[0])
test  = df[df["date"] >= cutoff_date]
LeakageChecker.check_split(train["date"], test["date"])

0.8050422546601431


In [4]:
df.groupby("location").corrwith(df["consumption"], numeric_only=True)

/var/folders/tq/8thty2lx4tg4sf4sj84788br0000gp/T/ipykernel_65895/1087572475.py:1: Pandas4Warning: DataFrameGroupBy.corrwith is deprecated
  df.groupby("location").corrwith(df["consumption"], numeric_only=True)


,consumption,temperature,hour,weekday,weekend,holiday,consumption_6days_ago,consumption_7days_ago,consumption_14days_ago,min_consumption_last_available_day,max_consumption_last_available_day,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m
location,,,,,,,,,,,,,,,,
bergen,1.0,-0.781353,0.134147,0.036712,0.031540,-0.066396,0.957196,0.955853,0.945482,0.931854,0.946835,0.020536,-0.044222,0.185993,0.073922,-0.241548
helsingfors,1.0,-0.600614,0.015791,-0.031094,-0.056871,0.078160,0.863168,0.854744,0.838041,0.831377,0.862453,0.019041,-0.041070,0.119613,0.112283,-0.152617
oslo,1.0,-0.861854,0.103920,0.024051,0.021060,-0.057859,0.961353,0.959324,0.945183,0.943334,0.956179,-0.005320,-0.070081,0.200496,-0.030781,-0.191553
stavanger,1.0,-0.781747,0.111166,0.027890,0.022275,-0.057846,0.962154,0.960346,0.946148,0.943597,0.956868,0.065873,0.017536,0.175748,0.099631,-0.250208
tromsø,1.0,-0.863436,0.158005,0.028138,0.023563,-0.043201,0.947569,0.948093,0.931008,0.918376,0.933007,-0.045262,-0.121273,0.176188,0.040997,0.024936
trondheim,1.0,-0.837779,0.148727,0.043482,0.039074,-0.056617,0.941949,0.940872,0.934927,0.911194,0.928386,-0.073843,-0.145197,0.185790,0.124168,-0.078619


In [ ]:
feats = ["location", "hour", "weekday", "temperature",
         "consumption_6days_ago", "consumption_7days_ago", "consumption_14days_ago",
         "max_consumption_last_available_day", "min_consumption_last_available_day"]
#TODO: use all avilable features, available 5 days ahead of time
LeakageChecker.check_feature_names(feats)

model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                         enable_categorical=True, tree_method="hist", random_state=42)
model.fit(train[feats], train["consumption"])
pred = model.predict(test[feats])
print("XGB:", mean_absolute_error(test["consumption"], pred))

mae = mean_absolute_error(test["consumption"], pred)
print("Naive 7d MAE:", mae, "MWH")
#MAE% (nMAE): MAE relative to mean consumption, so locations of different size can be compared
print("Naive 7d MAE%:", 100 * mae / test["consumption"].mean())

test = test.assign(pred=pred, err=lambda d: (d["consumption"] - d["pred"]).abs())
test.groupby("location")["err"].mean()

XGB: 0.37210498569928296


location
bergen         0.228170
helsingfors    0.145069
oslo           0.978012
stavanger      0.549945
tromsø         0.108931
trondheim      0.222659
Name: err, dtype: float64

In [ ]:
#Compare all models on the same hours: the test rows where every lag exists
scored = test.dropna(subset=feats)
validator = (ForecastValidator(scored["consumption"], groups=scored["location"])
             .add("Same hour last week", scored["consumption_7days_ago"])
             .add("Linear regression", pd.Series(lin_reg.predict(Xte), index=test_lr.index)[scored.index])
             .add("XGBoost", scored["pred"]))
validator.summary()

Er det bedre med jevnere feilfordeling, hvis kostnad ved imbalanse er eksponentiell?